# Partie 2 - OCR : Reconnaissance de caractères manuscrits

## 0. Imports

In [ ]:
import struct
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from collections import Counter

## 1. DATASET - Chargement des données

In [ ]:
class MyDataset(Dataset):
    def __init__(self, images_path, labels_path, mapping_path="image_data/mapping.txt"):
        # Chargement des images 
        with open(images_path, "rb") as f:
            magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
            images = np.frombuffer(f.read(), dtype=np.uint8)
            images = images.reshape(num, rows, cols)

        # Chargement des labels 
        with open(labels_path, "rb") as f:
            magic, num = struct.unpack(">II", f.read(8))
            labels = np.frombuffer(f.read(), dtype=np.uint8)

        # Chargement du mapping index → caractère 
        self.mapping = {}
        with open(mapping_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                idx_str, ascii_str = line.split(" ")
                self.mapping[int(idx_str)] = chr(int(ascii_str))

        # Correction de l'orientation (images pivotées dans EMNIST) 
        images = np.transpose(images, (0, 2, 1)).copy()

        # Normalisation : pixels entre 0 et 1 
        self.images = torch.tensor(images, dtype=torch.float32) / 255.0

        # Labels numériques (pour la loss) et caractères (pour affichage) 
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.char_labels = [self.mapping[int(l)] for l in labels]

        # Nombre de classes 
        self.num_classes = len(self.mapping)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Retourne l'image avec une dimension canal (1, 28, 28) et son label numérique
        return self.images[idx].unsqueeze(0), self.labels[idx]

    def show_sample(self, idx=0, count=10):
        """Affiche count images à partir de l'index idx"""
        count = min(count, len(self) - idx)
        cols = min(5, count)
        rows = (count + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
        axes = np.array(axes).reshape(-1)
        for i in range(count):
            img, lbl = self.images[idx + i], self.char_labels[idx + i]
            axes[i].imshow(img.numpy(), cmap='gray')
            axes[i].set_title(f"'{lbl}'", fontsize=12)
            axes[i].axis('off')
        for i in range(count, len(axes)):
            axes[i].axis('off')
        plt.suptitle("Exemples d'images du dataset", fontsize=14)
        plt.tight_layout()
        plt.show()

## 2. EDA - Exploration du dataset

In [ ]:
def explore_dataset(dataset):
    """Analyse exploratoire du dataset OCR"""

    print("=" * 50)
    print("EDA — Dataset OCR")
    print("=" * 50)
    print(f"Nombre total d'images     : {len(dataset)}")
    print(f"Taille des images         : 28 x 28 pixels")
    print(f"Nombre de classes         : {dataset.num_classes}")
    print(f"Caractères reconnus       : {list(dataset.mapping.values())}")

    # Distribution des classes 
    counts = Counter(dataset.char_labels)
    chars = list(counts.keys())
    freqs = list(counts.values())

    plt.figure(figsize=(14, 4))
    plt.bar(chars, freqs, color='steelblue')
    plt.title("Distribution des classes (nombre d'images par caractère)")
    plt.xlabel("Caractère")
    plt.ylabel("Nombre d'images")
    plt.tight_layout()
    plt.show()

    # Vérification du déséquilibre
    min_count = min(freqs)
    max_count = max(freqs)
    print(f"\nClasse la moins représentée : {min_count} images")
    print(f"Classe la plus représentée  : {max_count} images")
    if max_count / min_count > 2:
        print("Dataset déséquilibré")
    else:
        print("Dataset relativement équilibré")

    # Afficher quelques exemples 
    # Fonctionne avec MyDataset ou un Subset
    print("\nExemples d'images :")
    base = dataset.dataset if hasattr(dataset, 'dataset') else dataset
    base.show_sample(idx=0, count=10)

## 3. Préparation des données - Split train/val/test + DataLoaders

In [ ]:
def prepare_dataloaders(dataset, batch_size=64, train_ratio=0.7, val_ratio=0.15):
    """
    Divise le dataset en 3 parties :
    - Train (70%)     : le modèle apprend ici
    - Validation (15%): on surveille l'overfitting pendant l'entraînement
    - Test (15%)      : évaluation finale, on y touche qu'une seule fois
    """
    total = len(dataset)
    n_train = int(total * train_ratio)
    n_val = int(total * val_ratio)
    n_test = total - n_train - n_val

    train_set, val_set, test_set = random_split(
        dataset, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)  # Reproductible
    )

    print(f"Train      : {len(train_set)} images ({train_ratio*100:.0f}%)")
    print(f"Validation : {len(val_set)} images ({val_ratio*100:.0f}%)")
    print(f"Test       : {len(test_set)} images ({(1-train_ratio-val_ratio)*100:.0f}%)")

    # DataLoader : charge les données en batches, mélange le train
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


## 4. Modèle - CNN (Convolutional Neural Network)

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()

        # 3 blocs convolutifs 
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),   # Normalise les activations → entraînement plus stable
            nn.ReLU(),
            nn.MaxPool2d(2)       # 28×28 → 14×14
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)       # 14×14 → 7×7
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
            # Pas de pooling ici pour garder assez de spatial info
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.classifier(x)
        return x

## 5. Entraînement

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=15, lr=0.001):
    """
    Boucle d'entraînement complète avec suivi de la loss et de l'accuracy.

    Pour chaque epoch :
      1. Forward pass : image → modèle → prédiction
      2. Calcul de la loss (CrossEntropy)
      3. Backward pass : calcul des gradients
      4. Optimizer step : mise à jour des poids
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Entraînement sur : {device}")
    model = model.to(device)

    # CrossEntropyLoss : loss standard pour la classification multi-classes
    loss_fn = nn.CrossEntropyLoss()

    # Adam : optimizer adaptatif, converge rapidement
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Historique pour les courbes d'apprentissage
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [],  "val_acc": []
    }

    for epoch in range(num_epochs):
        # Phase d'entraînement 
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()          # Réinitialise les gradients
            outputs = model(images)        # Forward pass
            loss = loss_fn(outputs, labels)# Calcul de la loss
            loss.backward()                # Backward pass
            optimizer.step()               # Mise à jour des poids

            train_loss += loss.item()
            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        # Phase de validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():  # Pas de calcul de gradient en validation
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = loss_fn(outputs, labels)

                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        # Calcul des métriques
        train_acc = train_correct / train_total
        val_acc   = val_correct / val_total
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss / len(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch+1:2d}/{num_epochs}] "
              f"| Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} "
              f"| Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

    return model, history, device

## 6. Courbes d'apprentissage - Analyse overfitting/underfitting

In [ ]:
def plot_learning_curves(history):
    """
    Affiche les courbes de loss et d'accuracy pour détecter l'overfitting.

    Overfitting  : train_acc >> val_acc (le modèle mémorise au lieu d'apprendre)
    Underfitting : les deux sont faibles (le modèle est trop simple)
    Bon modèle   : les deux courbes sont proches et élevées
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history["train_loss"]) + 1)

    # Loss
    ax1.plot(epochs, history["train_loss"], label="Train", marker='o')
    ax1.plot(epochs, history["val_loss"],   label="Validation", marker='o')
    ax1.set_title("Loss au cours de l'entraînement")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)

    # Accuracy
    ax2.plot(epochs, history["train_acc"], label="Train", marker='o')
    ax2.plot(epochs, history["val_acc"],   label="Validation", marker='o')
    ax2.set_title("Accuracy au cours de l'entraînement")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

## 7. Évaluation finale - Matrice de confusion + métrique

In [ ]:
def evaluate_model(model, test_loader, mapping, device):
    """
    Évaluation finale sur le test set.
    - Accuracy globale
    - Matrice de confusion
    - Rapport détaillé par classe
    """
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    # Accuracy globale
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n{'='*50}")
    print(f"RÉSULTATS FINAUX SUR LE TEST SET")
    print(f"{'='*50}")
    print(f"Accuracy globale : {acc:.4f} ({acc*100:.2f}%)")

    # Noms des classes pour l'affichage
    class_names = [mapping[i] for i in sorted(mapping.keys())]

    # Rapport détaillé (precision, recall, f1 par classe)
    print("\nRapport par classe :")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # Matrice de confusion
    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(14, 12))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title("Matrice de confusion — OCR", fontsize=14)
    plt.xlabel("Classe prédite")
    plt.ylabel("Vraie classe")
    plt.tight_layout()
    plt.show()

    # Identifier les pires confusions
    np.fill_diagonal(cm, 0)  # On ignore la diagonale (bonne prédiction)
    top_errors = np.unravel_index(np.argsort(cm.ravel())[-5:], cm.shape)
    print("\nTop 5 des confusions les plus fréquentes :")
    for i, j in zip(top_errors[0][::-1], top_errors[1][::-1]):
        print(f"  '{class_names[i]}' prédit comme '{class_names[j]}'")

    return acc

## 8. Main - Rassemblement final

In [ ]:
if __name__ == "__main__":

    images_file  = "image_data/train-images-idx3-ubyte"
    labels_file  = "image_data/train-labels-idx1-ubyte"
    mapping_path = "image_data/mapping.txt"

    # Chargement
    print("Chargement du dataset...")
    full_dataset = MyDataset(images_file, labels_file, mapping_path)

    # EDA sur le dataset complet avant de faire le sous-échantillonnage
    print("\nAnalyse exploratoire du dataset complet :")
    explore_dataset(full_dataset)

    # Sous-échantillonnage à 20% — ~139 586 images, amplement suffisant
    subset_size = int(0.20 * len(full_dataset))
    indices = torch.randperm(len(full_dataset), generator=torch.Generator().manual_seed(42))[:subset_size]
    subset_indices = indices.tolist()

    # Créer un dataset réduit proprement sans Subset
    full_dataset.images     = full_dataset.images[subset_indices]
    full_dataset.labels     = full_dataset.labels[subset_indices]
    full_dataset.char_labels = [full_dataset.char_labels[i] for i in subset_indices]
    dataset = full_dataset

    print(f"Sous-ensemble utilisé : {len(dataset)} images (20% du dataset)")


    # Splits
    print("\nPréparation des splits train/val/test...")
    train_loader, val_loader, test_loader = prepare_dataloaders(
        dataset, batch_size=128
    )

    # Modèle
    model = CNN(num_classes=dataset.num_classes)

    # Entraînement court
    print("\nDémarrage de l'entraînement...")
    model, history, device = train_model(
        model, train_loader, val_loader,
        num_epochs=10,
        lr=0.001
    )

    # Résultats
    plot_learning_curves(history)
    evaluate_model(model, test_loader, dataset.mapping, device)

    torch.save(model.state_dict(), "ocr_model.pth")
    print("\nModèle sauvegardé !")